LangGraph Agents

## Why LangGraph?

We previously built an agent manually:
```
User
 ↓
LLM
 ↓
Does the LLM want a tool?
 ↓
Yes
 ↓
Execute tool
 ↓
Send result back to LLM
 ↓
LLM
 ↓
Final answer
```
Our Python code had to manage:

- Conversation state
- LLM calls
- Tool execution
- Tool results
- Loops
- Deciding when to stop

As the agent becomes more complex, this control flow becomes difficult to manage.

LangGraph provides a framework for explicitly representing this workflow as a **graph**.

---

# 1. The Core Idea

LangGraph represents an agent as:

```text
        ┌──────────────┐
        │     START    │
        └──────┬───────┘
               ↓
        ┌──────────────┐
        │     Agent    │
        │     (LLM)    │
        └──────┬───────┘
               ↓
          Tool needed?
          /          \
        YES           NO
         ↓             ↓
   ┌──────────┐    ┌──────────┐
   │   Tools  │    │    END   │
   └────┬─────┘    └──────────┘
        │
        │
        └──────────────→ Agent

# 2. What Is a Graph?

A graph consists mainly of:

## Nodes

A node represents some piece of work.

For example:

-- LLM node
-- Tool node
-- Validation node
-- Database node
-- Human approval node

## Edges

An edge defines what happens next.

For example:
```
LLM → Tools
```
or:
```
Tools → LLM
```
## Conditional Edges

A conditional edge allows the workflow to make a decision.

For example:
```
LLM
 ↓
Tool required?
 ├── YES → Tools
 └── NO  → END

 ```

# 3. What Is State?

State is the information that moves through the graph.

For an agent, state commonly contains the conversation:
```
state = {
    "messages": [...]
}
```
** Conceptually:** 

```

             STATE
               │
               ▼
        ┌─────────────┐
        │    Agent    │
        └──────┬──────┘
               │
               ▼
        Updated STATE
               │
               ▼
        ┌─────────────┐
        │    Tools    │
        └──────┬──────┘
               │
               ▼
        Updated STATE
               │
               ▼
             Agent
```
The state is what allows different nodes to communicate.

# 4. LangGraph's Three Fundamental Concepts

The most important concepts to understand are:
```
State
Nodes
Edges
```

Think about them like this:
```
State  = Data
Node   = Work
Edge   = Control flow
```

For example:

                 STATE
                   │
                   ▼
              ┌─────────┐
              │   LLM   │ ← Node
              └────┬────┘
                   │
              Conditional
                 Edge
                   │
             ┌─────┴─────┐
             ↓           ↓
          ┌──────┐     END
          │ Tool │
          └──┬───┘
             │
             ↓
            LLM

# 5. Our Previous Agent vs LangGraph

** Manual Agent **

We wrote:
```
while True:

    response = client.responses.create(...)

    if tool_call:
        execute_tool()

    else:
        return final_answer
```

The while True represented the agent loop.

** LangGraph Agent **

LangGraph represents the same idea explicitly:

```

START
  ↓
Agent
  ↓
Should call tool?
  ↓
Tools
  ↓
Agent
  ↓
Should call tool?
  ↓
END
```
Therefore LangGraph is not fundamentally giving the LLM some magical new intelligence.

It gives us a structured execution engine for agent workflows.

# Why a Graph Instead of a Simple Loop?

A simple agent might look like:
```
LLM → Tool → LLM → Tool → LLM
```

But real systems can become much more complicated:

                 ┌─────────────┐
                 │     LLM     │
                 └──────┬──────┘
                        ↓
                 Need retrieval?
                  /           \
                YES            NO
                 ↓              ↓
              Search          Validate
                 ↓              ↓
              Rerank          Human Review
                 ↓              ↓
                LLM           Approve?
                 │           /       \
                 │         YES        NO
                 │          ↓          ↓
                 └────────→ LLM      END
                              ↓
                            END

Representing this with nested while, if, and try/except blocks becomes difficult.

A graph makes the workflow explicit.

# 7. LangGraph Agent Mental Model

Think of LangGraph as:
```

                    LANGGRAPH
                       │
             ┌─────────┴─────────┐
             │                   │
           State               Graph
                                 │
                       ┌─────────┴─────────┐
                       │                   │
                     Nodes               Edges
                       │                   │
                  "Do work"          "Go somewhere"
```
The LLM is usually one of the nodes.

The LLM itself is not the graph.

The graph controls the execution around the LLM.

# 8. Important Distinction

Do not think:

LangGraph = Agent

A better mental model is:
```
LangGraph
    ↓
Workflow / orchestration framework
    ↓
Can be used to build agents
```

An agent can be implemented using LangGraph, but LangGraph can also represent workflows that aren't really autonomous agents.

For example:
```
Input
 ↓
Validate
 ↓
Transform
 ↓
Store
 ↓
END
```

This is a graph/workflow, but it doesn't necessarily involve an agent.

# 9. LangGraph Agent Architecture

A typical tool-using LangGraph agent looks like:
```

                    USER
                      │
                      ▼
               ┌────────────┐
               │    State   │
               └─────┬──────┘
                     ↓
               ┌────────────┐
               │    Agent   │
               │    Node    │
               │    (LLM)   │
               └─────┬──────┘
                     ↓
              ┌──────────────┐
              │ Conditional  │
              │    Edge      │
              └──────┬───────┘
                     │
             ┌───────┴───────┐
             ↓               ↓
       Tool required       No tool
             ↓               ↓
       ┌──────────┐        END
       │   Tool   │
       │   Node   │
       └────┬─────┘
            │
            ↓
          State
            │
            └──────────→ Agent
```